In [45]:
# libraries
import pandas as pd
import sqlite3

In [46]:
# This creates the database file if it doesn't exist
conn = sqlite3.connect("school.db")
cur = conn.cursor()


1.Creating students table

In [47]:
create_table_query = """CREATE TABLE IF NOT EXISTS Students(
    StudentID INT PRIMARY KEY,
    Name VARCHAR(50),
    Age INT,
    City VARCHAR(50));"""

cur.execute(create_table_query);


In [48]:
# Check if table exists
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
tables


,name
0,Students
1,Courses
2,Enrollments


In [37]:
cur.execute("""SELECT name FROM sqlite_master WHERE type = 'table';""")
# Fetch the result and store it in table_names
table_names = cur.fetchall()
table_names

[('Students',), ('Courses',), ('Enrollments',)]

1.2 Inserting data to student table

In [6]:
cur.execute("""
INSERT INTO Students (StudentID, Name, Age, City) VALUES
(1, 'Wanjiru', 20, 'Nairobi'),
(2, 'Kamau', 22, 'Mombasa'),
(3, 'Achieng', 21, 'Kisumu'),
(4, 'Otieno', 23, 'Nakuru'),
(5, 'Mwende', 19, 'Eldoret');
""")
conn.commit()

2.Creating Courses table

In [7]:
create_table_query = """ 
CREATE TABLE IF NOT EXISTS Courses (
    CourseID INT PRIMARY KEY,
    CourseName VARCHAR(100),
    Credits INT
);
"""
cur.execute(create_table_query)
conn.commit()

2.2 Inserting data to courses table

In [8]:
cur.execute("""
INSERT INTO Courses (CourseID, CourseName, Credits) VALUES
(101, 'SQL Basics', 3),
(102, 'Data Analysis', 4),
(103, 'Web Development', 5),
(104, 'Machine Learning', 4);
""")
conn.commit()

3. Creating Enrollment table

In [9]:
create_table_query = """ 
CREATE TABLE Enrollments (
    EnrollmentID INT PRIMARY KEY,
    StudentID INT,
    CourseID INT,
    EnrollmentDate DATE,
    FOREIGN KEY (StudentID) REFERENCES Students(StudentID),
    FOREIGN KEY (CourseID) REFERENCES Courses(CourseID)
);
"""
cur.execute(create_table_query)
conn.commit()

3.2 Inserting data to enrollment table

In [10]:
cur.execute("""
INSERT INTO Enrollments (EnrollmentID, StudentID, CourseID, EnrollmentDate) VALUES
(1, 1, 101, '2023-01-15'),
(2, 1, 102, '2023-01-16'),
(3, 2, 101, '2023-01-17'),
(4, 3, 103, '2023-01-18'),
(5, 4, 104, '2023-01-19'),
(6, 5, 102, '2023-01-20');
""")
conn.commit()

In [15]:
#inner Join
q="""SELECT* FROM Students s INNER JOIN Enrollments e
ON s.StudentID=e.StudentID;"""
pd.read_sql(q,conn)

,StudentID,Name,Age,City,EnrollmentID,StudentID,CourseID,EnrollmentDate
0,1,Wanjiru,20,Nairobi,1,1,101,2023-01-15
1,1,Wanjiru,20,Nairobi,2,1,102,2023-01-16
2,2,Kamau,22,Mombasa,3,2,101,2023-01-17
3,3,Achieng,21,Kisumu,4,3,103,2023-01-18
4,4,Otieno,23,Nakuru,5,4,104,2023-01-19
5,5,Mwende,19,Eldoret,6,5,102,2023-01-20


In [32]:
#left Join
q='''SELECT * FROM Students s LEFT JOIN Enrollments e ON s.StudentID=e.StudentID;'''
pd.read_sql(q,conn)

,StudentID,Name,Age,City,EnrollmentID,StudentID,CourseID,EnrollmentDate
0,1,Wanjiru,20,Nairobi,1,1,101,2023-01-15
1,1,Wanjiru,20,Nairobi,2,1,102,2023-01-16
2,2,Kamau,22,Mombasa,3,2,101,2023-01-17
3,3,Achieng,21,Kisumu,4,3,103,2023-01-18
4,4,Otieno,23,Nakuru,5,4,104,2023-01-19
5,5,Mwende,19,Eldoret,6,5,102,2023-01-20


In [38]:
#Right Join
q='''SELECT * FROM Students s RIGHT JOIN Enrollments e ON s.StudentID=e.StudentID;'''
pd.read_sql(q,conn)

DatabaseError: Execution failed on sql 'SELECT * FROM Students s RIGHT JOIN Enrollments e ON s.StudentID=e.StudentID;': RIGHT and FULL OUTER JOINs are not currently supported

In [40]:
#Inner Join of three tables
q='''SELECT s.Name, c.CourseName FROM Students s INNER JOIN Enrollments e ON s.StudentID=e.StudentID
INNER JOIN Courses c  ON e.CourseID=c.CourseID '''
pd.read_sql(q,conn)

,Name,CourseName
0,Wanjiru,SQL Basics
1,Wanjiru,Data Analysis
2,Kamau,SQL Basics
3,Achieng,Web Development
4,Otieno,Machine Learning
5,Mwende,Data Analysis


In [43]:
#Full outer Join students table and enrollments table
#Using UNION ALL 
q='''SELECT * FROM Students s LEFT JOIN Enrollments e ON s.StudentID=e.StudentID UNION ALL 
SELECT * FROM Enrollments e LEFT JOIN Students s ON s.StudentID=e.StudentID;'''
pd.read_sql(q,conn)

,StudentID,Name,Age,City,EnrollmentID,StudentID,CourseID,EnrollmentDate
0,1,Wanjiru,20,Nairobi,1,1,101,2023-01-15
1,1,Wanjiru,20,Nairobi,2,1,102,2023-01-16
2,2,Kamau,22,Mombasa,3,2,101,2023-01-17
3,3,Achieng,21,Kisumu,4,3,103,2023-01-18
4,4,Otieno,23,Nakuru,5,4,104,2023-01-19
5,5,Mwende,19,Eldoret,6,5,102,2023-01-20
6,1,1,101,2023-01-15,1,Wanjiru,20,Nairobi
7,2,1,102,2023-01-16,1,Wanjiru,20,Nairobi
8,3,2,101,2023-01-17,2,Kamau,22,Mombasa
9,4,3,103,2023-01-18,3,Achieng,21,Kisumu


In [44]:
q = '''
         SELECT *
         FROM Students s
         LEFT JOIN Enrollments e 
         ON s.StudentID = e.StudentID
         UNION ALL
         SELECT *
         FROM Students s
         RIGHT JOIN Enrollments e 
         ON s.StudentID = e.StudentID;
    '''
pd.read_sql(q, conn)

DatabaseError: Execution failed on sql '
         SELECT *
         FROM Students s
         LEFT JOIN Enrollments e 
         ON s.StudentID = e.StudentID
         UNION ALL
         SELECT *
         FROM Students s
         RIGHT JOIN Enrollments e 
         ON s.StudentID = e.StudentID;
    ': RIGHT and FULL OUTER JOINs are not currently supported